# F27 Scale-Dependence Validation — FL 5f + CA/TX 1f with B3 + `next_gru` cat head

Purpose
-------
Resolve the scale-dependent flag raised by F32 (FL 1-fold): on AL/AZ (5f) the `next_gru` cat head lifts cat F1 by +2-3 pp (p=0.0312 Wilcoxon on AZ), but on FL at n=1 the direction flipped by ~0.9 pp. Three runs here settle the question:

1. **F33 — FL 5f × 50ep B3+next_gru**: decisive σ on FL. Commits the north-star Path A (universal next_gru) vs Path B (scale-dependent).
2. **F34 — CA upstream pipeline + CA 1f B3+next_gru**: first CA data point AND scale check at ~500K check-ins.
3. **F35 — TX upstream pipeline + TX 1f B3+next_gru**: first TX data point.

Budget (Colab T4 / A100)
------------------------
- FL 5f × 50ep: ~3–5 h (on T4) for B3-gru; ~1–2 h on A100.
- CA upstream (embedding + transition + parquets): ~2–4 h.
- CA 1f × 50ep B3-gru: ~1–2 h.
- TX upstream + 1f: same as CA.
- **Total:** 10–18 h on T4; 4–8 h on A100.

Prereqs
-------
- Google Drive with the raw Gowalla check-ins (`California.parquet`, `Texas.parquet`, `Florida.parquet`) and the TIGER shapefiles for CA, TX, FL.
- Default Drive layout (matches `notebooks/colab_all_states_parallel_embeddings.ipynb`):
  ```
  MyDrive/mestrado/PoiMtlNet/
    data/
      checkins/
        California.parquet
        Texas.parquet
        Florida.parquet
      misc/
        tl_2022_06_tract_CA/tl_2022_06_tract.shp
        tl_2022_48_tract_TX/tl_2022_48_tract.shp
        tl_2022_12_tract_FL/tl_2022_12_tract.shp
    output/       # Check2HGI embeddings land here (will be populated for CA/TX)
    results/      # Training results land here
  ```
- **Florida-specific note**: if you've already trained Check2HGI on FL locally, you can copy `output/check2hgi/florida/` into your Drive's `output/` to skip F33's upstream step. The notebook detects and skips if present.

At the end, paste the `Summary Table` cell's output back into the chat for analysis.


---
## ① Bootstrap — mount Drive, clone repo, install deps


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# --- Config: adjust if your Drive layout differs ---
DRIVE_ROOT = Path('/content/drive/MyDrive/mestrado/PoiMtlNet')
DATA_ROOT = DRIVE_ROOT / 'data'
OUTPUT_DIR = DRIVE_ROOT / 'output'
RESULTS_ROOT = DRIVE_ROOT / 'results'

for p in [DATA_ROOT, OUTPUT_DIR, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DATA_ROOT'] = str(DATA_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_DIR)
os.environ['RESULTS_ROOT'] = str(RESULTS_ROOT)
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # harmless on CUDA

print(f'DATA_ROOT:    {DATA_ROOT}')
print(f'OUTPUT_DIR:   {OUTPUT_DIR}')
print(f'RESULTS_ROOT: {RESULTS_ROOT}')


In [ ]:
# Clone the branch with B3 + F20 + F27 cat-head + p1 GETNext-hard support
BRANCH = 'worktree-check2hgi-mtl'
REPO_DIR = Path('/content/PoiMtlNet')

if not REPO_DIR.exists():
    !git clone -q https://github.com/VitorHugoOli/PoiMtlNet.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch --all -q
!git checkout {BRANCH}
!git pull --ff-only
!git log --oneline -5


In [ ]:
# Install pinned deps — mirrors notebooks/colab_all_states_parallel_embeddings.ipynb
!pip install -q -r requirements_colab.txt
!pip install -q cvxpy torch-geometric pytorch_warmup torchmetrics fvcore

import torch
!pip uninstall torch-scatter torch-sparse torch-cluster -y
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install -q torch-sparse  -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install -q torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html

print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available(), '| device =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Shared launch env — every training / pipeline call below inherits this.
import subprocess

ENV = {
    **os.environ,
    'PYTHONPATH': 'src',
    'DATA_ROOT': str(DATA_ROOT),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'RESULTS_ROOT': str(RESULTS_ROOT),
    'PYTORCH_ENABLE_MPS_FALLBACK': '1',
}

def run(cmd: list[str], log_file: str | None = None) -> int:
    print(f'$ {" ".join(cmd)}')
    if log_file:
        with open(log_file, 'w') as f:
            p = subprocess.run(cmd, cwd=REPO_DIR, env=ENV, stdout=f, stderr=subprocess.STDOUT)
        print(f'  (log: {log_file}, exit={p.returncode})')
        if p.returncode != 0:
            !tail -30 {log_file}
    else:
        p = subprocess.run(cmd, cwd=REPO_DIR, env=ENV)
    return p.returncode


---
## ② Sanity check — confirm data availability


In [ ]:
# Raw check-in parquets (required for all three states)
for state_title in ['California', 'Texas', 'Florida']:
    p = DATA_ROOT / 'checkins' / f'{state_title}.parquet'
    print(f'  checkins/{state_title}.parquet: {"OK" if p.exists() else "MISSING"}  ({p})')

# TIGER shapefiles
TIGER = [
    ('California', 'tl_2022_06_tract_CA', 'tl_2022_06_tract.shp'),
    ('Texas',      'tl_2022_48_tract_TX', 'tl_2022_48_tract.shp'),
    ('Florida',    'tl_2022_12_tract_FL', 'tl_2022_12_tract.shp'),
]
for state_title, dirname, filename in TIGER:
    p = DATA_ROOT / 'misc' / dirname / filename
    print(f'  misc/{dirname}/{filename}: {"OK" if p.exists() else "MISSING"}  ({p})')

# Existing Check2HGI outputs (will be populated for CA/TX)
for state in ['california', 'texas', 'florida']:
    d = OUTPUT_DIR / 'check2hgi' / state
    has_emb = (d / 'embeddings.parquet').exists() or (d / 'input' / 'next.parquet').exists()
    has_nr = (d / 'input' / 'next_region.parquet').exists()
    has_trans = (d / 'region_transition_log.pt').exists()
    print(f'  check2hgi/{state}: embeddings={has_emb} · next_region.parquet={has_nr} · region_transition_log={has_trans}')


---
## ③ F33 — FL 5f × 50ep B3 + next_gru

Runs first because FL's upstream pipeline is usually already on your Drive (from prior work). If missing, the next cell detects and runs the upstream steps for FL too.

In [ ]:
# Step 3a — Upstream pipeline helper (idempotent; skips if artefacts already present)
#
# For each state we need:
#   1. output/check2hgi/<state>/embeddings.parquet  (+ input/next.parquet)  — produced by Check2HGI training
#   2. output/check2hgi/<state>/region_transition_log.pt                     — produced by scripts/compute_region_transition.py
#   3. output/check2hgi/<state>/input/next_region.parquet (with last_region_idx)
#
# FL artefacts usually already live in your Drive from prior work; this helper
# only does what's missing. CA + TX are full runs (no prior data).

import textwrap

_RUN_ONE_STATE_HELPER = textwrap.dedent(r"""
    # Helper run inside a subprocess to avoid importing `pipelines.embedding.check2hgi.pipe`
    # (the dotted filename is not a valid Python module path).
    import sys, importlib.util
    sys.path.insert(0, 'src')
    sys.path.insert(0, 'research')
    from copy import copy
    from configs.paths import Resources, EmbeddingEngine
    from embeddings.check2hgi.check2hgi import create_embedding
    from data.inputs.builders import generate_next_input_from_checkins

    # Load CONFIG from the pipeline script by path (filename has a dot — no direct import)
    spec = importlib.util.spec_from_file_location('c2h_pipe', 'pipelines/embedding/check2hgi.pipe.py')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)

    STATE_TITLE = sys.argv[1]      # e.g. 'California'
    STATE_LOWER = sys.argv[2]      # e.g. 'california'
    TL_MAP = {
        'alabama':   Resources.TL_AL,
        'arizona':   Resources.TL_AZ,
        'florida':   Resources.TL_FL,
        'california': Resources.TL_CA,
        'texas':     Resources.TL_TX,
    }

    cfg = copy(mod.CONFIG)
    cfg.shapefile = TL_MAP[STATE_LOWER]
    print(f'[{STATE_TITLE}] Check2HGI training + next input generation...', flush=True)
    create_embedding(state=STATE_TITLE, args=cfg)
    generate_next_input_from_checkins(STATE_TITLE, EmbeddingEngine.CHECK2HGI)
    print(f'[{STATE_TITLE}] done.', flush=True)
""").strip()

HELPER_SCRIPT = '/tmp/c2h_one_state.py'
with open(HELPER_SCRIPT, 'w') as f:
    f.write(_RUN_ONE_STATE_HELPER)


def ensure_check2hgi_pipeline(state_lower: str, state_title: str) -> None:
    out = OUTPUT_DIR / 'check2hgi' / state_lower
    emb_ok = (out / 'embeddings.parquet').exists() or (out / 'input' / 'next.parquet').exists()
    trans_ok = (out / 'region_transition_log.pt').exists()
    parquet_ok = (out / 'input' / 'next_region.parquet').exists()

    if not emb_ok:
        print(f'[{state_lower}] Step 1 — Check2HGI embeddings + next.parquet (long run: 2–4 h on T4)')
        rc = run(['python', HELPER_SCRIPT, state_title, state_lower],
                 log_file=f'/tmp/c2h_{state_lower}.log')
        if rc != 0:
            raise RuntimeError(f'Check2HGI failed for {state_lower} — see /tmp/c2h_{state_lower}.log')
    else:
        print(f'[{state_lower}] Step 1 skipped — embeddings already present')

    if not trans_ok:
        print(f'[{state_lower}] Step 2 — region_transition_log')
        rc = run(['python', 'scripts/compute_region_transition.py', '--state', state_lower])
        if rc != 0:
            raise RuntimeError(f'compute_region_transition failed for {state_lower}')
    else:
        print(f'[{state_lower}] Step 2 skipped — region_transition_log.pt already present')

    if not parquet_ok or True:  # Always re-check: last_region_idx schema may be older
        print(f'[{state_lower}] Step 3 — ensure next_region.parquet has last_region_idx')
        rc = run(['python', 'scripts/regenerate_next_region.py', '--state', state_lower])
        if rc != 0:
            raise RuntimeError(f'regenerate_next_region failed for {state_lower}')

    print(f'[{state_lower}] upstream pipeline OK\n')


# Ensure FL data is ready (skipped if already there)
ensure_check2hgi_pipeline('florida', 'Florida')


In [ ]:
# Step 3b — FL 5f × 50ep B3+gru
FL_TRANSITION = str(OUTPUT_DIR / 'check2hgi' / 'florida' / 'region_transition_log.pt')

run([
    'python', '-u', 'scripts/train.py',
    '--state', 'florida',
    '--task', 'mtl', '--task-set', 'check2hgi_next_region', '--engine', 'check2hgi',
    '--folds', '5', '--epochs', '50', '--seed', '42',
    '--task-a-input-type', 'checkin', '--task-b-input-type', 'region',
    '--model', 'mtlnet_crossattn',
    '--mtl-loss', 'static_weight', '--category-weight', '0.75',
    '--cat-head', 'next_gru',
    '--reg-head', 'next_getnext_hard',
    '--reg-head-param', 'd_model=256', '--reg-head-param', 'num_heads=8',
    '--reg-head-param', f'transition_path={FL_TRANSITION}',
    '--max-lr', '0.003', '--gradient-accumulation-steps', '1', '--no-checkpoints',
], log_file='/tmp/f33_fl_5f_b3_gru.log')


---
## ④ F34 — CA upstream + 1f B3 + next_gru


In [ ]:
ensure_check2hgi_pipeline('california', 'California')


In [ ]:
CA_TRANSITION = str(OUTPUT_DIR / 'check2hgi' / 'california' / 'region_transition_log.pt')

run([
    'python', '-u', 'scripts/train.py',
    '--state', 'california',
    '--task', 'mtl', '--task-set', 'check2hgi_next_region', '--engine', 'check2hgi',
    '--folds', '1', '--epochs', '50', '--seed', '42',
    '--task-a-input-type', 'checkin', '--task-b-input-type', 'region',
    '--model', 'mtlnet_crossattn',
    '--mtl-loss', 'static_weight', '--category-weight', '0.75',
    '--cat-head', 'next_gru',
    '--reg-head', 'next_getnext_hard',
    '--reg-head-param', 'd_model=256', '--reg-head-param', 'num_heads=8',
    '--reg-head-param', f'transition_path={CA_TRANSITION}',
    '--max-lr', '0.003', '--gradient-accumulation-steps', '1', '--no-checkpoints',
], log_file='/tmp/f34_ca_1f_b3_gru.log')


---
## ⑤ F35 — TX upstream + 1f B3 + next_gru


In [ ]:
ensure_check2hgi_pipeline('texas', 'Texas')


In [ ]:
TX_TRANSITION = str(OUTPUT_DIR / 'check2hgi' / 'texas' / 'region_transition_log.pt')

run([
    'python', '-u', 'scripts/train.py',
    '--state', 'texas',
    '--task', 'mtl', '--task-set', 'check2hgi_next_region', '--engine', 'check2hgi',
    '--folds', '1', '--epochs', '50', '--seed', '42',
    '--task-a-input-type', 'checkin', '--task-b-input-type', 'region',
    '--model', 'mtlnet_crossattn',
    '--mtl-loss', 'static_weight', '--category-weight', '0.75',
    '--cat-head', 'next_gru',
    '--reg-head', 'next_getnext_hard',
    '--reg-head-param', 'd_model=256', '--reg-head-param', 'num_heads=8',
    '--reg-head-param', f'transition_path={TX_TRANSITION}',
    '--max-lr', '0.003', '--gradient-accumulation-steps', '1', '--no-checkpoints',
], log_file='/tmp/f35_tx_1f_b3_gru.log')


---
## ⑥ Summary — paste this cell's output back into the chat


In [ ]:
import json, glob

def fmt(v):
    if isinstance(v, dict):
        m, s = v.get('mean', 0), v.get('std', 0)
        if s:
            return f'{m*100:6.3f} ± {s*100:5.3f}'
        return f'{m*100:6.3f}'
    return f'{v:6.3f}' if isinstance(v, (int, float)) else str(v)

STATES = [('florida', '5f'), ('california', '1f'), ('texas', '1f')]

print(f'{"State":10s}  {"protocol":8s}  {"cat F1":16s}  {"cat Acc@1":16s}  {"reg Acc@10":16s}  {"reg Acc@5":16s}  {"reg MRR":16s}')
print('-'*120)

for state, proto in STATES:
    pattern = str(RESULTS_ROOT / 'check2hgi' / state / 'mtlnet_*_ep50_*' / 'summary' / 'full_summary.json')
    runs = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    if not runs:
        print(f'{state:10s}  {proto:8s}  (no result yet)')
        continue
    d = json.load(open(runs[0]))
    c, r = d.get('next_category', {}), d.get('next_region', {})
    print(f'{state:10s}  {proto:8s}  {fmt(c.get("f1",0)):16s}  {fmt(c.get("accuracy",0)):16s}  '
          f'{fmt(r.get("top10_acc_indist",0)):16s}  {fmt(r.get("top5_acc_indist",0)):16s}  '
          f'{fmt(r.get("mrr_indist",0)):16s}')

print()
print('Baseline reference (from prior MTL-B3 results):')
print('  AL 5f B3+gru  cat_F1=0.4271 ± 0.0137  reg_Acc10=0.5960 ± 0.0409')
print('  AZ 5f B3+gru  cat_F1=0.4581 ± 0.0130  reg_Acc10=0.5382 ± 0.0311')
print('  FL 1f B3+gru  cat_F1=0.6572           reg_Acc10=0.6526         (from F32 on M4 Pro)')
print('  FL 1f B3+default (pre-F27) cat_F1 mean 0.6665 reg_Acc10 mean 0.6619')

print()
print('Paste the above table back into the chat. I will analyse and decide Path A vs B for NORTH_STAR.')


---
## ⑦ Optional — copy full_summary JSONs back to the repo

If you want me to see the JSONs directly instead of just the table, this cell copies them to a single folder and tars it for download.


In [ ]:
OUT = Path('/tmp/f27_results'); OUT.mkdir(exist_ok=True)
for state in ['florida', 'california', 'texas']:
    pattern = str(RESULTS_ROOT / 'check2hgi' / state / 'mtlnet_*_ep50_*' / 'summary' / 'full_summary.json')
    runs = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    if runs:
        target = OUT / f'{state}_full_summary.json'
        !cp {runs[0]} {target}
        print(f'{state}: copied  {runs[0]} → {target}')

!cd /tmp && tar czf f27_results.tar.gz f27_results/
!ls -la /tmp/f27_results.tar.gz

# Download in Colab:
from google.colab import files
files.download('/tmp/f27_results.tar.gz')
